<a href="https://colab.research.google.com/github/prasa129/Math/blob/main/Matrix_Subspaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Matrix Subspaces

## 9-26-25

This notebook provides a lightweight OOP wrapper around an $m \times n$ numpy array $A$. It computes the four fundamental subspaces: image, kernel, co-image, and co-kernel, with no dependencies aside from Numpy.  

For $A \in \mathbb{R}^{m \times n}$, rank$(A)=r$:

- Image:
$$
\text{img}(A) = \{Ax: x \in \mathbb{R}^{n}\} ⊆ \mathbb{R}^{m}, \text{ dim img}(A)=r
$$
- Coimage:
$$
\text{coimg}(A) = \text{img}(A^{T})  = \{A^{T}y: y \in \mathbb{R}^{m}\} ⊆ \mathbb{R}^{n}, \text{ dim coimg}(A)=r
$$
- Kernel:
$$
\text{ker}(A) = \{ x \in \mathbb{R}^{n}: Ax = 0 \}, \text{ dim ker}(A)=n-r
$$
- Cokernel:
$$
\text{coker}(A) = \{ y \in \mathbb{R}^{m}: A^{T}y = 0 \}, \text{ dim coker}(A)=m-r
$$

Rank-nullity:

$$
\text{dim img}(A) + \text{dim ker}(A) = n
$$
$$
\text{dim coimg}(A) + \text{dim coker}(A) = m
$$

In [1]:
import numpy as np

class Matrix:
    """
    Minimal, readable OOP wrapper around an m x n numpy array A.
    Compute basis for four fundamental subspaces using simple REF/RREF
    helpers.

    Parameters
    ----------
    A : array_like (m, n)
        Input matrix.
    tol : float, optional
        Numerical tolerance to treat small values as zero (default 1e-12).
    """
    def __init__(self, A, tol=1e-12):
        self.A   = np.array(A, dtype=float, copy=True)
        self.tol = tol

    # ---------- elementary matrix utilities ----------
    @staticmethod
    def perm_matrix(m, i, j):
        """
        Build the m × m permutation matrix that swaps rows i and j.
        i, j use 0 based indexing.
        Args:
          m : int, dimension of the identity matrix
          i, j : int, 0-based row indices to swap
        Returns:
          E : np.ndarray, m x m permutation matrix
        """
        #Initialize I_m, E
        I = np.identity(m)
        E = I.copy()

        #Swap rows
        E[[i, j]] = I[[j, i]]
        return E

    @staticmethod
    def elem_matrix(m, a, j, i):
        """
        Build the m × m elementary matrix that multiplies row j by a
        Args:
          m : int, dimension of identity matrix
          a : float, scalar
          j, i : int, 0-based row indices
        Returns:
          E : np.ndarray, m x m elementary matrix
        """
        #Init E, set E_ji = a
        E = np.identity(m)
        E[j, i] = a
        return E

    @staticmethod
    def swapper(P, L, U, i, k):
        """
        Swap rows i and k in P, L, U. Swap lower triangular rows i and k in L.
        Args:
          P, L : np.ndarray, m x m matrices
          U : np.ndarray, m x n matrix
          i, k : int, 0-based row indices
        Returns:
          P, L, U : np.ndarray, m x m and m x n matrices
        Row swap by left multiplying by elementary permutation matrix.
        """
        #Initialize I_m, elementary swap matrix E
        m = P.shape[0]
        I = np.identity(m)
        E = Matrix.perm_matrix(m, i, k)

        # Swap rows of P and U
        P = E @ P
        U = E @ U

        #Swap only L's elements below diagonal
        L = E @ (L - I) + I

        #Return tuple of matrices
        return P, L, U

    # ---------- REF/RREF ----------
    def REF(self):
        """
        Perform permuted LU factorization of A.
        Args:
        Returns:
          P : np.ndarray, m x m permutation matrix
          L : np.ndarray, m x m lower triangular matrix
          U : np.ndarray, m x n upper triangular matrix
        Returns P, L, U, with PA = LU, U in REF.
        """
        #Copy A as U
        U = self.A.copy()
        m, n = U.shape

        #Initialize P, L as I_m
        L = np.identity(m)
        P = np.identity(m)

        #Iterate through min(m,n)
        r = min(m, n)
        for i in range(r):

            #If pivot is 0, search rows below for non-zero element
            if U[i, i] == 0:

                #Indexer and boolean flag to avoid repeated searches
                k = i + 1
                swapped = False

                #Search column elements in rows below 0 pivot
                while k < m and not swapped:

                    #If pivot found, swap rows
                    if U[k, i] != 0:

                        #Update P, L, U with swap utility
                        P, L, U = Matrix.swapper(P, L, U, i, k)
                        swapped = True
                    else:
                        k += 1

                #If all of col i is 0 from row i down, skip the pivot column
                if not swapped and U[i, i] == 0:
                    continue

            #Forward elimination for col i. Iterate through rows below row i.
            for j in range(i + 1, m):

                #Modify rows if there's a sub-diagonal entry in col i
                if U[j, i] != 0:

                    #Compute multiplier
                    a = U[j, i] / U[i, i]

                    #Perform row operation Row_j = Row_j - a*Row_i
                    U[j] = U[j] - a * U[i]

                    #Record row operation in L
                    L = L @ Matrix.elem_matrix(m, a, j, i)

        #Return tuple of matrices
        return P, L, U

    def RREFify(self, U):
        """
        Bottom-up sweep to convert REF matrix U into RREF.
        Args:
          U : np.ndarray, m x n matrix
        Returns:
          V : np.ndarray, m x n matrix
        Normalize pivot rows by dividing by pivot elements.
        """
        #Initialize RREF form V as copy of REF U
        V = U.copy()
        m, n = V.shape

        #Iterate through rows, bottom to top
        for i in range(m - 1, -1, -1):

            #Find indices of non-zero elements in row i
            nz = np.where(np.abs(V[i]) > self.tol)[0]

            #If no pivot found, continue to next row
            if nz.size == 0:
                continue

            #First index is the pivot
            pivot_idx = int(nz[0])

            #Extract pivot
            piv = V[i, pivot_idx]

            #Skip pivot if within tol of 0
            if np.abs(piv) <= self.tol:
                continue

            #Normalize i'th row by dividing through by pivot
            V[i, :] = V[i, :] / piv

            #Clear out column above pivot
            for j in range(i):
                V[j, :] = V[j, :] - V[j, pivot_idx] * V[i, :]

        #Control for numerical imprecision in RREF V
        V[np.abs(V) < self.tol] = 0.0

        #Return V, the RREF of A
        return V

    def RREF(self):
        """
        Wrapper: compute RREF(A) using REF -> RREFify.
        """
        #Discard P,L, retain U from REF()
        _, _, U = self.REF()

        #Compute RREF
        V = self.RREFify(U)

        #Return RREF of A
        return V

    # ---------- Four fundamental subspaces ----------
    # ---------- img A, coimg A, ker A, coker A ------
    def image(self):
        """
        Compute basis for img(A) (the column space), return
        basis as m x r matrix B. Select pivot columns of A
        based on pivot columns of RREF(A).
        Args:
        Returns:
          B : np.ndarray, m x r matrix with basis for img(A)
        Note, r = rank(A).
        """
        #RREF of A
        V = self.RREF()
        m, n = self.A.shape

        #Track pivot column indices
        pivot_cols = []

        #Iterate through rows
        for i in range(m):

            #Identify col indices of row i's non-zero elements
            nz = np.where(np.abs(V[i]) > self.tol)[0]

            #If row isn't all 0s, append first index for pivot
            if nz.size > 0:
                pivot_cols.append(int(nz[0]))

        #If no pivot cols, A must be 0 matrix
        if len(pivot_cols) == 0:

            #return 0_m vector as basis
            return np.zeros((m, 0))


        #B is m x r (number of rows in A by A's rank)
        B = self.A[:, pivot_cols].copy()

        #Return B, basis for image of A
        return B

    def row_space(self):
        """
        Basis for row(A) constructed from non-zero rows of
        RREF(A). Function returns R with shape (r x n).
        Args:
        Returns:
          R : np.ndarray, r x n matrix with basis for row(A)
        Note, img(A^T) returns an n x r matrix whose columns
        are a basis for A. This approach returns a reduced
        basis, where each vector is a row in R^n.
        """
        #RREF of A
        V = self.RREF()

        #Select rows of RREF(A) with any non-zero elements (within tolerance)
        rows = [V[i].copy() for i in range(V.shape[0])
                if np.any(np.abs(V[i]) > self.tol)]

        #If all rows are 0, return 0_n row vector as basis
        if not rows:
            R = np.zeros((0, V.shape[1]))
        else:
            #Otherwise, vertically stack lin. indep. rows of RREF(A)
            R = np.vstack(rows)

        #Return basis R, an r x n matrix
        return R

    def kernel(self):
        """
        Compute basis for the kernel of A using free variable = 1
        construction from RREF(A).
        Args:
        Returns:
          C : np.ndarray, n x (n-r) matrix with basis for ker(A)
        """
        #RREF of A
        V = self.RREF()
        m, n = self.A.shape

        #Track pivot row and col indices
        pivot_rows, pivot_cols = [], []

        #Iterate through rows of RREF A
        for i in range(m):

            #Indices of V's row i's non-zero elements
            nz = np.where(np.abs(V[i]) > self.tol)[0]

            #Store pivot index
            if nz.size > 0:
                pivot_rows.append(i)
                pivot_cols.append(int(nz[0]))

        #If all columns are free, A is 0 matrix
        if len(pivot_cols) == 0:

            #A = 0_mxn => kernel is R^n with basis I_n
            C = np.eye(n)
            return C

        #Indices of free variable columns (non pivot columns)
        free_cols = [j for j in range(n) if j not in pivot_cols]

        #nullity of A: n - rank(A)
        rnull = len(free_cols)

        #Init C = 0_nx(n-r)
        C = np.zeros((n, rnull))

        """
        Construct the basis. In RREF, each pivot row r
        satisfies x_p + sum_j in free V[r,j]x_j = 0
        for p the row's pivot column. For free column f,
        x_f = 1, x_j = 0 for j =/= f, thus x_p = -V[r,f]
        """
        #Iterate over free variable columns
        for k, f in enumerate(free_cols):

            #Init X as vector 0_n
            x = np.zeros(n)

            #Set chosen free variable x_f as 1
            x[f] = 1.0

            #Solve each pivot variable x_p from pivot row
            for row, p in zip(pivot_rows, pivot_cols):
                x[p] = -V[row, f]

            #Store null vector as C's column k
            C[:, k] = x

        #Return basis for nullspace
        return C

    def co_kernel(self):
        """
        Compute basis for co-kernel of A using kernel of A^T.
        Args:
        Returns:
          D : np.ndarray, m x (m-r) matrix with basis for co-kernel of A.
        r is the rank of A.
        """
        #coker = ker(A^T)
        D = Matrix(self.A.T, tol=self.tol).kernel()

        #Post-process trivial coker
        if D.shape[1] == 0:
          m = self.A.shape[0]
          D = np.zeros(m)
        return D

    # ---------- Convenience summaries ----------
    def rank(self):
        """
        Compute rank(A) using RREF(A).
        Args:
        Returns:
          r : int, rank of A
        """
        #dim img(A)
        r = self.image().shape[1]
        return r

    def dims_summary(self):
        """
        Return a dictionary with the dimensions of the four fundamental subspaces
        and check rank-nullity relationships.
        Args:
        Returns:
          dict : dictionary with dimensions of fundamental subspaces
        """
        #A's dimensions and rank r
        m, n = self.A.shape
        r = self.rank()

        #dimension of kernel
        nullity = self.kernel().shape[1]

        #dimension of co-kernel,
        try:
            left_nullity = self.co_kernel().shape[1]
        #Trivial co-ker has dimension of 0
        except IndexError:
            left_nullity = 0

        #Dictionary verifying rank-nullity relations
        return {"m": m,
                "n": n,
                "rank": r,
                "nullity (n-r)": nullity,
                "left nullity (m-r)": left_nullity,
                "rank + nullity = n": (r + nullity == n),
                "rank + left nullity = m": (r + left_nullity == m)
                }

I demonstrate the class with a test matrix.

A=\begin{bmatrix}
1 & 2 & -1 & 1\\
2 & 5 & \ \ 0 & 4\\
-1& -3& \ \ 1 & 0
\end{bmatrix}


$\xrightarrow{\,R_2\leftarrow R_2-2R_1,\; R_3\leftarrow R_3+R_1\,}$
\begin{bmatrix}
1 & 2 & -1 & 1\\
0 & 1 & 2 & 2\\
0 & -1& 0 & 1
\end{bmatrix}


$\xrightarrow{\,R_3\leftarrow R_3+R_2\,}$
\begin{bmatrix}
1 & 2 & -1 & 1\\
0 & 1 & 2 & 2\\
0 & 0 & 2 & 3
\end{bmatrix}

$\xrightarrow{\,R_3\leftarrow \tfrac12 R_3\,}$
\begin{bmatrix}
1 & 2 & -1 & 1\\
0 & 1 & 2 & 2\\
0 & 0 & 1 & \tfrac32
\end{bmatrix}

$\xrightarrow{\,R_2\leftarrow R_2-2R_3,\; R_1\leftarrow R_1+R_3\,}$
\begin{bmatrix}
1 & 2 & 0 & \tfrac52\\
0 & 1 & 0 & -1\\
0 & 0 & 1 & \tfrac32
\end{bmatrix}

$\xrightarrow{\,R_1\leftarrow R_1-2R_2\,}$

$\operatorname{RREF}(A)=$
\begin{bmatrix}
1 & 0 & 0 & \tfrac{9}{2}\\
0 & 1 & 0 & -1\\
0 & 0 & 1 & \tfrac{3}{2}
\end{bmatrix}

Columns 1, 2, and 3 have pivots. A basis for img($A$) is given by:

$$
\text{span}\left\{
\begin{bmatrix}1\\[2pt]2\\[-2pt]-1\end{bmatrix},
\begin{bmatrix}2\\[2pt]5\\[-2pt]-3\end{bmatrix},
\begin{bmatrix}-1\\[2pt]0\\[0pt]1\end{bmatrix}
\right\}.
$$

The nonzero rows of RREF(A) form a basis for the row space:
$$\text{span}\left\{
\begin{bmatrix}1&0&0&\tfrac{9}{2}\end{bmatrix},
\begin{bmatrix}0&1&0&-1\end{bmatrix},
\begin{bmatrix}0&0&1&\tfrac{3}{2}\end{bmatrix}
\right\}.
$$

From RREF(A),
\begin{align*}
x_1+\tfrac{9}{2}x_4&=0 \\
x_2-x_4&=0 \\
x_3+\tfrac{3}{2}x_4&=0 \\
\end{align*}

Let $x_{4}=t$. Then
$$
x=t\begin{bmatrix}-\tfrac{9}{2}\\[2pt]1\\[-2pt]-\tfrac{3}{2}\\[2pt]1\end{bmatrix},
\qquad
\text{ker}(A)=\operatorname{span}\!\left\{\begin{bmatrix}-\tfrac{9}{2}\\[2pt]1\\[-2pt]-\tfrac{3}{2}\\[2pt]1\end{bmatrix}\right\}
$$

By rank-nullity,

\begin{align*}
\text{rank}(A) &= r = 3 \\
\text{dim coimg}(A) &= r = 3 \\
\text{dim coimg}(A) + \text{dim coker}(A) &= m = 3 \\
\text{dim coker}(A) &= m - r = 0 \\
 \implies \text{coker}(A) &= \{ 0 \}
\end{align*}

because the only subspace of $\mathbb{R}^{m}$ with dimension 0 is the zero-vector: $\{0\}$.

In [2]:
if __name__ == "__main__":

    #A is a 3 x 4 example matrix
    A = np.array([[1, 2, -1,  1],
                  [2, 5,  0,  4],
                  [-1,-3,  1, 0]], dtype=float)

    #Test Matrix class functionality
    M = Matrix(A)

    #A and RREF(A)
    print("A =\n", M.A)
    print("\nRREF(A) =\n", M.RREF())

    #Image, co-image, kernel, and co-kernel
    B = M.image()
    R = M.row_space()
    C = M.kernel()
    D = M.co_kernel()

    #Bases for fundamental subspaces
    print("\nBasis for column space (im A): B shape =", B.shape, "\n", B)
    print("\nBasis for row space (row A):   R shape =", R.shape, "\n", R)
    print("\nBasis for kernel (ker A):      C shape =", C.shape, "\n", C)
    print("\nBasis for co-kernel (ker(A^T)):D shape =", D.shape, "\n", D)

    #Verify bases are valid
    print("\nCheck bases for subspaces are valid:")
    print("  rank(A) =", M.rank())
    print("  A @ C ≈ 0 ? ", np.allclose(M.A @ C, 0))
    print("  A.T @ D ≈ 0 ? ", np.allclose(M.A.T @ D, 0))
    print("\nDimension summary:")
    for k,v in  M.dims_summary().items():
      print(f" {k}: {v}")

A =
 [[ 1.  2. -1.  1.]
 [ 2.  5.  0.  4.]
 [-1. -3.  1.  0.]]

RREF(A) =
 [[ 1.   0.   0.   4.5]
 [ 0.   1.   0.  -1. ]
 [ 0.   0.   1.   1.5]]

Basis for column space (im A): B shape = (3, 3) 
 [[ 1.  2. -1.]
 [ 2.  5.  0.]
 [-1. -3.  1.]]

Basis for row space (row A):   R shape = (3, 4) 
 [[ 1.   0.   0.   4.5]
 [ 0.   1.   0.  -1. ]
 [ 0.   0.   1.   1.5]]

Basis for kernel (ker A):      C shape = (4, 1) 
 [[-4.5]
 [ 1. ]
 [-1.5]
 [ 1. ]]

Basis for co-kernel (ker(A^T)):D shape = (3,) 
 [0. 0. 0.]

Check bases for subspaces are valid:
  rank(A) = 3
  A @ C ≈ 0 ?  True
  A.T @ D ≈ 0 ?  True

Dimension summary:
 m: 3
 n: 4
 rank: 3
 nullity (n-r): 1
 left nullity (m-r): 0
 rank + nullity = n: True
 rank + left nullity = m: True
